# LAB 4: Architecting Advanced Retrieval-Augmented Generation (RAG) Systems for Structured Screenplays

You are a Lead AI Architect at Leavesden Studios, the production powerhouse behind the Harry Potter film franchise. The studio is currently developing a massive, interactive 25th-Anniversary Digital Archive, allowing fans, VFX artists, and film historians to query the original production screenplay of Harry Potter and the Sorcerer's Stone to understand the director's vision.


To power this experience, the engineering team deployed a standard Naive RAG (Retrieval-Augmented Generation) pipeline using basic semantic embeddings to search the screenplay script.


It was a disaster. During a pre-launch demo, the system failed critically in two ways:


- Screenplays are highly structured, spatial documents. The naive text chunker blindly sliced through the script every 500 characters. It frequently severed a Character's Name from their actual Dialogue block. When the RAG pipeline retrieved the orphaned dialogue and fed it to the Director, the AI had no idea who was speaking. It confidently stated that Hagrid was giving Dumbledore's emotional speech about the Mirror of Erised.

- When a production designer asked a thematic question like, "Summarize Harry's emotional state when he receives his wand," the dense embedding model performed beautifully, retrieving the exact scene at Ollivanders. However, when a continuity editor searched for a precise alphanumeric location tag like "Scene 165 - INT. TRAIN COMPARTMENT" or an exact fictional incantation like "Oculus Reparo", the semantic vector space failed completely. The dense embeddings blurred the specific jargon into generic concepts of "trains" and "broken glass," retrieving entirely irrelevant scenes.

Compounding the issue, the naive brute-force search (O(N)) began to severely lag when scaled up to include thousands of concurrent queries from fans, threatening the real-time interactivity needed for the exhibit.


## Dataset

A screenplay.pdf file is provided to you containing the "Harry Potter and the Sorceror's Stone" screenplay.


The document includes the dialogues and interactions between the various characters throughout the film.

## PHASES OF THE LAB:

#### 0. Install necessary packages
#### 1. Data Extraction using Document Loaders
#### 2. Chunking
#### 3. Indexing, Embeddings and storing in a Vector DB
#### 4.1. Retrieval, LLM Integration, Chains and Response generation
#### 4.2. How to add a memory component? (Chat History)
#### 4.3. Custom Branching Logic & Anti-Hallucination
#### 5.1. Hybrid Search
#### 5.2. Re-Ranking
#### 5.3. Basic Query Expansion
#### 6.1. HyDE (Hypothetical Document Embeddings)
#### 6.2. RAG-Fusion
#### 6.3. CRAG (Corrective RAG)
#### 7.1 Dynamic IR Evaluation Metrics
#### 7.2 TruLens Evaluation

## PHASE 0: Install necessary packages

In [9]:
%pip install langchain_community langchain_docling langchain_text_splitters langchain_huggingface langchain_chroma pypdf pymupdf

## PHASE 1: Data Extraction using Document Loaders

A Document Loader is the very first step in building a RAG application..

LLMs only understand raw text, but your data is likely trapped in various formats like PDFs, Word documents, websites, YouTube transcripts, or databases.

A Document Loader is a specialized tool that connects to these sources, extracts the information, and converts it into a standardized Document object.

Every Document object contains two main parts:

- page_content: The actual extracted text from the file.

- metadata: A dictionary containing information about the data (e.g., source file name, page number, date created).

Refer to source: https://docs.langchain.com/oss/python/integrations/document_loaders

This link contains all document loaders that can be used as part of Langchain.

Please scroll down to PDFs since our dataset is a PDF.

You can explore the rest on your own if interested.

#### You will need to extract the text and store it in form of a Document object.
#### You will implement 2 ways: Normal text extraction and text to Markdown.

#### For Normal text extraction, Use any 2 of the following loaders: PyPDF, PyMuPDF, PDFPlumber
#### For text to Markdown, We will be using Docling.

#### You will need to research all of these different document loaders to find out which is most apt for our usecase (A PDF containing only text).

A few links to help you:
1. https://www.geeksforgeeks.org/python/introduction-to-python-pypdf2-library/
2. https://pymupdf.readthedocs.io/en/latest/about.html
3. https://www.pdfplumber.com/#Features
4. https://www.docling.ai/
5. https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/

#### You can also use AI to evaluate which one is better for the usecase.

#### IMPORTANT NOTE: You will need to install the corresponding package according to which document loader you want to use:
#### For example, pypdf - pip install pypdf.

In [2]:
PDF_PATH = '/content/screenplay.pdf'

### Helper for normal text extraction:

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

def langchain_extract_pymupdf(file_path):
    print("Extracting using LangChain PyMuPDFLoader")
    try:
        # Initialize loader
        loader = PyMuPDFLoader(file_path)

        # Load documents
        documents = loader.load()
        print(f"Total Pages: {len(documents)}")

        print("\nExtracted Text:")

        first_3_pages_text = ""

        # Print first 3 pages
        for i in range(min(3, len(documents))):
            page_text = documents[i].page_content
            print(f"\n--- Page {i+1} ---\n{page_text}")
            first_3_pages_text += f"\nPage {i+1}:\n{page_text}\n"

        return documents

    except Exception as e:
        print(f"Error extracting with PyMuPDFLoader: {e}")


# Call function
pymupdf_docs = langchain_extract_pymupdf(PDF_PATH)

Extracting using LangChain PyMuPDFLoader
Error extracting with PyMuPDFLoader: File path /content/screenplay.pdf is not a valid file or url


### Helper for text to Markdown:

In [4]:
%pip install hf_xet langchain_docling

In [5]:
import logging
from langchain_docling import DoclingLoader
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Some arguments to control the PDF extraction process. Adjusted to run faster.
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True

custom_converter = DocumentConverter(
    format_options={
        "pdf": PdfFormatOption(pipeline_options=pipeline_options)
    }
)

def extract_with_docling(file_path):
    print("Extracting using DoclingLoader:")
    try:
        # Pass the custom converter to the loader
        loader = DoclingLoader(
            file_path=file_path,
            export_type="markdown",
            converter=custom_converter
        )

        documents = loader.load()

        print(f"Total Pages: {len(documents)}")
        print("Extracted Markdown Content:")

        first_3_pages_text = ""

        # Print first 3 pages
        for i in range(min(3, len(documents))):
            page_text = documents[i].page_content
            print(f"\n--- Page {i+1} ---\n{page_text}")
            first_3_pages_text += f"\nPage {i+1}:\n{page_text}\n"

        return documents

    except Exception as e:
        print(f"Error extracting with Docling: {e}")


# Call function
docling_docs = extract_with_docling(PDF_PATH)

Extracting using DoclingLoader:
Error extracting with Docling: [Errno 2] No such file or directory: '/content/screenplay.pdf'


In [10]:
docling_docs = extract_with_docling(PDF_PATH)

Extracting using DoclingLoader:
Error extracting with Docling: [Errno 2] No such file or directory: 'screenplay.pdf'


## PHASE 2: Chunking


Chunking is the process of breaking down large text into smaller segments called chunks.

It’s an essential preprocessing technique that helps optimize the relevance of the content ultimately stored in a vector database. The trick lies in finding chunks that are big enough to contain meaningful information, while small enough to enable performant applications and low latency responses for workloads such as retrieval augmented generation and agentic workflows.



In [6]:
%pip install langchain-text-splitters tiktoken

### Different chunking strategies:

#### Fixed-size chunking

This is the most common and straightforward approach to chunking: we simply decide the number of tokens in our chunk, and use this number to break up our documents into fixed size chunks. Usually, this number is the max context window size of the embedding model (such as 1024 for llama-text-embed-v2, or 8196 for text-embedding-3-small). Keep in mind that different embedding models may tokenize text differently, so you will need to estimate token counts accurately.


#### Content-aware Chunking

Although fixed-size chunking is quite easy to implement, it can ignore critical structure within documents that can be used to inform relevant chunks. Content-aware chunking refers to strategies that adhere to structure to help inform the meaning of our chunks.

#### Simple Sentence and Paragraph splitting

As we mentioned before, some embedding models are optimized for embedding sentence-level content. But sometimes, sentences need to be mined from larger text datasets that aren’t preprocessed. In these cases, it’s necessary to use sentence chunking, and there are several approaches and tools available to do this:

- Naive splitting: The most naive approach would be to split sentences by periods (“.”), new lines, or white space.
- NLTK: The Natural Language Toolkit (NLTK) is a popular Python library for working with human language data. It provides a trained sentence tokenizer that can split the text into sentences, helping to create more meaningful chunks.
- spaCy: spaCy is another powerful Python library for NLP tasks. It offers a sophisticated sentence segmentation feature that can efficiently divide the text into separate sentences, enabling better context preservation in the resulting chunks.


#### Recursive Character Level Chunking

LangChain implements a RecursiveCharacterTextSplitter that tries to split text using separators in a given order. The default behavior of the splitter uses the ["\n\n", "\n", " ", ""] separators to break paragraphs, sentences and words depending on a given chunk size.

This is a great middle ground between always splitting on a specific character and using a more semantic splitter, while also ensuring fixed chunk sizes when possible.

#### Token based Chunking

Token-based chunking splits text into fixed-size segments based on token count, aligning directly with how Large Language Models (LLMs) process input. This method ensures chunk sizes match the LLM’s context window constraints, improving retrieval accuracy and model compatibility.

#### Document structure-based chunking

When chunking large documents such as PDFs, DOCX, HTML, code snippets, Markdown files and LaTex, specialized chunking methods can help preserve the original structure of the content during chunk creation.


#### Semantic Chunking

Semantic chunking involves breaking a document into sentences, grouping each sentence with its surrounding sentences, and generating embeddings for these groups. By comparing the semantic distance between each group and its predecessor, you can identify where the topic or theme shifts, which defines the chunk boundaries.

#### Markdown Chunking

Markdown-based chunking is a strategy for splitting structured documents into smaller, semantically coherent segments while preserving hierarchical context.


#### You will need to decide on which output of a particular document loader for normal text extraction out of PyPDF, PyMuPDF, PDFPlumber.
#### Make this decision based on how best the text is being retrieved.

#### You will perform chunking for documents from natural text extraction and for markdown documents both.
#### Choose 2 chunking strategies for natural text extraction and implement them. Explain why you are using these strategies in the report.


1. Recursive Chunking (Natural Text)

In [ ]:
PDF_PATH = "screenplay.pdf"

In [17]:
from langchain_community.document_loaders import PyPDFLoader

def langchain_extract_pypdf(file_path):
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    return documents

pypdf_docs = langchain_extract_pypdf(PDF_PATH)

In [15]:
PDF_PATH = "screenplay.pdf"

In [18]:
recursive_chunks = chunk_recursive(pypdf_docs)

Chunking with RecursiveCharacterTextSplitter:
Created 432 chunks.


In [19]:
print(type(pypdf_docs))

<class 'list'>


In [12]:
if 'pypdf_docs' not in globals():
    print("Run extraction step first!")

Run extraction step first!


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_recursive(loaded_documents):
    print("Chunking with RecursiveCharacterTextSplitter:")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False
    )

    chunks = text_splitter.split_documents(loaded_documents)
    print(f"Created {len(chunks)} chunks.")
    return chunks


# Call
recursive_chunks = chunk_recursive(pypdf_docs)

Chunking with RecursiveCharacterTextSplitter:
Created 432 chunks.


2. Fixed-Size Chunking (Baseline)

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_fixed(loaded_documents):
    print("Chunking with Fixed-Size Strategy:")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False
    )

    chunks = text_splitter.split_documents(loaded_documents)
    print(f"Created {len(chunks)} chunks.")
    return chunks


# Call
fixed_chunks = chunk_fixed(pypdf_docs)

Chunking with Fixed-Size Strategy:
Created 674 chunks.


3. Markdown + Hybrid Chunking (IMPORTANT ⭐)

In [23]:
print(type(docling_docs))

<class 'NoneType'>


In [24]:
docling_docs = extract_with_docling(PDF_PATH)

Extracting using DoclingLoader:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total Pages: 1
Extracted Markdown Content:

--- Page 1 ---
-0-,:,

..

,----,:

.

'":-1

\_\_

.

.

.

:

':-:;.

'

..

\_.,,,.·

..

0

. ·

## HARRY  POTTER  AND THE  \_SORCERER'S  STONE

Screenplay  · by Steve Kloves

Based on the novel bv J.K.  Rowlino . "

- ·  Pink

| Shooting ·Draft BlueRevision ·Pink Revision YellowRevision · GreenRevision Gold Revision .Buff Revision. ·Salmon Revision Cherry Revision TanRevision 2m WhiteRevision 2m BlueRevision 2 rd Pink Revision   | 11/09/00 '12/09/00 22/09/00 13/10/00 16/10/00 31/10/00 03/11/00 14/11/00 06/12/00 08/01/01" 01/02/01 01/02/01 07/0ZJOl 18/04/01   |
|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------|

j

0

## 1-18 OMITTED  SCS. 1  18 &amp; PAG

In [25]:
print(len(docling_docs))

1


In [26]:
hybrid_chunks = chunk_markdown_hybrid(docling_docs)

Chunking with Markdown + Recursive Hybrid Strategy:
Created 482 hybrid chunks.


In [27]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def chunk_markdown_hybrid(markdown_documents):
    print("Chunking with Markdown + Recursive Hybrid Strategy:")

    headers = [
        ("#", "Header1"),
        ("##", "Header2"),
        ("###", "Header3"),
    ]

    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers)

    all_chunks = []

    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False
    )

    for doc in markdown_documents:
        md_chunks = md_splitter.split_text(doc.page_content)

        for chunk in md_chunks:
            sub_chunks = recursive_splitter.split_text(chunk.page_content)
            all_chunks.extend(sub_chunks)

    print(f"Created {len(all_chunks)} hybrid chunks.")
    return all_chunks


# Call
hybrid_chunks = chunk_markdown_hybrid(docling_docs)

Chunking with Markdown + Recursive Hybrid Strategy:
Created 482 hybrid chunks.


#### For Markdown chunking, we use MarkdownHeaderTextSplitter
#### We will implement a hybrid of this and RecursiveCharacterTextSplitter

In [28]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def chunk_markdown_hybrid(loaded_documents):
    print("Chunking with MarkdownHeaderTextSplitter + Recursive:")

    # Combine all markdown pages into one text
    full_markdown_text = "\n\n".join([doc.page_content for doc in loaded_documents])

    # Define the headers we want to split on
    headers_to_split_on = [
        ("#", "Header1"),
        ("##", "Header2"),
        ("###", "Header3"),
    ]

    # Perform the initial structural split
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False
    )

    md_header_splits = markdown_splitter.split_text(full_markdown_text)
    print(f"Stage 1: Created {len(md_header_splits)} structural chunks based on headers.")

    # Apply a size limit using the Recursive Splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        length_function=len,
        is_separator_regex=False
    )

    final_splits = text_splitter.split_documents(md_header_splits)

    print(f"Stage 2: Created {len(final_splits)} final size-controlled chunks.")

    return final_splits


# Call
hybrid_chunks = chunk_markdown_hybrid(docling_docs)

Chunking with MarkdownHeaderTextSplitter + Recursive:
Stage 1: Created 73 structural chunks based on headers.
Stage 2: Created 482 final size-controlled chunks.


In [29]:
hybrid_chunks_docling = chunk_markdown_hybrid(docling_docs)

Chunking with MarkdownHeaderTextSplitter + Recursive:
Stage 1: Created 73 structural chunks based on headers.
Stage 2: Created 482 final size-controlled chunks.


## PHASE 3: Indexing, Embeddings and Vector DB

#### Embeddings

Computers do not understand English, but they excel at math. Embeddings are a way to translate text (words, sentences, or whole documents) into arrays of numbers (vectors).

If two pieces of text have similar meanings (e.g., "puppy" and "dog"), their resulting number arrays will be located close to each other in a mathematical space. Generating these numbers is the job of an embedding model.

Choose Your Embedding Model:

For your code, you need to pick a model. Here are some of the best current options:

- BAAI/bge-m3 (Open Source): Highly capable, supports multiple languages, and handles long contexts well.

- nomic-embed-text (Open Source): Extremely efficient and powerful for its size; great for local deployments.

#### Vector DB

A standard database (like SQL) searches for exact keyword matches.

A Vector Database stores the numerical embeddings generated above and searches by similarity. When you ask a question, the database converts your question into a vector and finds the stored vectors that are mathematically closest to it (usually calculating the $L_2$ distance or Cosine Similarity).

- FAISS: Faiss (Facebook AI Similarity Search) is a library that allows developers to quickly search for embeddings of multimedia documents that are similar to each other. It solves limitations of traditional query search engines that are optimized for hash-based searches, and provides more scalable similarity search functions.

- Chroma DB: Chroma DB is an open-source vector database designed for efficiently storing, searching and managing vector embeddings which are numeric representations used in AI and machine learning for tasks like semantic search and recommendation systems. It enables fast similarity search and offers a simple API for developers making it well-suited for building and deploying AI-driven applications.

#### Indexing

If you have a million documents, calculating the distance between your query and every single document takes too long ($O(N)$ time complexity).

Indexing is the process of organizing those vectors into a smart data structure so you can find the nearest neighbors quickly without checking every single one.

Here are the four indexing methods you will implement:
- Flat Index (IndexFlatL2): No optimization. It literally compares your query to every single vector. It guarantees 100% perfect accuracy but is too slow for massive datasets. Think of it as brute-force searching.

- IVF Index (Inverted File): This method groups your vectors into clusters (Voronoi cells). When you search, it figures out which cluster your query is closest to, and only searches inside that specific cluster. It trades a tiny bit of accuracy for a huge speed boost.

- HNSW (Hierarchical Navigable Small World): This builds a multi-layered graph. It starts searching on a "highway" layer connecting distant points, and as it gets closer to the target, it drops down to local "street" layers to find the exact neighbors. It is currently the industry standard for combining high speed with high accuracy.

- PQ (Product Quantization): This is a compression technique. It breaks your massive vectors into smaller chunks and replaces them with shorthand codes. It drastically reduces the memory (RAM) needed to store the index, but lowers the accuracy of the search.

Refer:
1. https://www.pinecone.io/learn/series/faiss/hnsw/
2. https://www.ai-bites.net/rag-7-indexing-methods-for-vector-dbs-similarity-search/


In [30]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.6 MB/s eta 0:00:00


In [31]:
import faiss
import numpy as np
import time
import uuid
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

def advanced_faiss_indexing_with_huggingface(chunks):
    """
    Takes a list of LangChain Document chunks, embeds them, creates multiple
    FAISS indexes to benchmark them, and returns a LangChain FAISS VectorStore.
    """
    model_kwargs = {'device': 'cpu'}
    encode_kwargs = {'normalize_embeddings': True}

    # ✅ TODO 1: Initialize Embedding Model
    print("Loading embedding model...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3",
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwargs
    )

    print(f"Extracting texts and generating raw embeddings for {len(chunks)} chunks...")
    texts = [doc.page_content for doc in chunks]

    embeddings_list = embeddings.embed_documents(texts)
    embeddings_np = np.array(embeddings_list).astype('float32')

    d = embeddings_np.shape[1]
    print(f"Embedding dimension: {d}")

    # ✅ TODO 2: Flat Index
    print("Building Flat Index...")
    index_flat = faiss.IndexFlatL2(d)
    index_flat.add(embeddings_np)

    # ✅ TODO 3: IVF Index
    print("Building IVF Index...")
    nlist = 5
    quantizer = faiss.IndexFlatL2(d)
    index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist)
    index_ivf.train(embeddings_np)
    index_ivf.add(embeddings_np)

    # ✅ TODO 4: HNSW Index
    print("Building HNSW Index...")
    index_hnsw = faiss.IndexHNSWFlat(d, 16)
    index_hnsw.add(embeddings_np)

    # ✅ TODO 5: PQ Index
    print("Building PQ Index...")
    m = 32  # subquantizers
    index_pq = faiss.IndexPQ(d, m, 8)
    index_pq.train(embeddings_np)
    index_pq.add(embeddings_np)

    # 🔹 BENCHMARKING
    print("\nRetrieval Benchmarks:")
    query = "Who gives Harry his first birthday cake?"
    query_emb = np.array([embeddings.embed_query(query)]).astype('float32')
    k = 3

    def benchmark_faiss(index, index_name):
        start_time = time.time()
        distances, indices = index.search(query_emb, k)
        latency = (time.time() - start_time) * 1000
        print(f"{index_name} Latency: {latency:.4f} ms | Retrieved Indices: {indices[0]}")

    benchmark_faiss(index_flat, "Flat Index (L2) ")
    benchmark_faiss(index_ivf, "IVF Index       ")
    benchmark_faiss(index_hnsw, "HNSW Index      ")
    benchmark_faiss(index_pq, "PQ Index        ")

    # ✅ TODO 6: Wrap HNSW into LangChain FAISS
    print("\nWrapping HNSW index into LangChain FAISS VectorStore:")

    docstore = InMemoryDocstore()
    index_to_docstore_id = {}

    for i, doc in enumerate(chunks):
        doc_id = str(uuid.uuid4())
        docstore.add({doc_id: doc})
        index_to_docstore_id[i] = doc_id

    vector_db = FAISS(
        embedding_function=embeddings,
        index=index_hnsw,
        docstore=docstore,
        index_to_docstore_id=index_to_docstore_id
    )

    # ✅ TODO 7: Save locally
    vector_db.save_local("faiss_index")

    print("VectorStore saved locally as 'faiss_index'")

    return vector_db

In [32]:
hybrid_vector_db = advanced_faiss_indexing_with_huggingface(hybrid_chunks)

Loading embedding model...
Extracting texts and generating raw embeddings for 482 chunks...
Embedding dimension: 1024
Building Flat Index...
Building IVF Index...
Building HNSW Index...
Building PQ Index...

Retrieval Benchmarks:
Flat Index (L2)  Latency: 2.6469 ms | Retrieved Indices: [98 99 63]
IVF Index        Latency: 0.4303 ms | Retrieved Indices: [ 98  99 198]
HNSW Index       Latency: 0.2532 ms | Retrieved Indices: [98 99 63]
PQ Index         Latency: 0.5550 ms | Retrieved Indices: [ 98  99 284]

Wrapping HNSW index into LangChain FAISS VectorStore:
VectorStore saved locally as 'faiss_index'


In [33]:
%pip install langchain-chroma

In [34]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import uuid

def embed_docling_with_nomic(hybrid_markdown_chunks):
    print("Embedding Docling Markdown with Nomic:")

    # ✅ Initialize Nomic embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name="nomic-ai/nomic-embed-text-v1",
        model_kwargs={"device": "cpu"},   # change to "cuda" if GPU available
        encode_kwargs={"normalize_embeddings": True}
    )

    persist_dir = "./chroma_docling_nomic"

    try:
        print(f"Embedding {len(hybrid_markdown_chunks)} chunks:")

        # Generate IDs for each document
        ids = [str(uuid.uuid4()) for _ in hybrid_markdown_chunks]

        # ✅ Create Chroma vector database
        vector_db = Chroma.from_documents(
            documents=hybrid_markdown_chunks,
            embedding=embeddings,
            ids=ids,
            persist_directory=persist_dir,
            collection_name="docling_nomic_collection"
        )

        print(f"Successfully embedded and saved Chroma database to {persist_dir}")
        return vector_db

    except Exception as e:
        print(f"Error during Nomic embedding: {e}")

In [35]:
embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1",
    model_kwargs={
        "device": "cpu",
        "trust_remote_code": True   # ✅ THIS FIXES IT
    },
    encode_kwargs={"normalize_embeddings": True}
)

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/nomic-ai/nomic-bert-2048:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [36]:
import os
os.environ["HF_ALLOW_CODE_EVAL"] = "1"   # 🔥 important fix

embeddings = HuggingFaceEmbeddings(
    model_name="nomic-ai/nomic-embed-text-v1",
    model_kwargs={
        "device": "cpu",
        "trust_remote_code": True
    },
    encode_kwargs={
        "normalize_embeddings": True
    }
)

In [37]:
import os
import uuid
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# 🔥 VERY IMPORTANT (must be BEFORE model load)
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

def embed_docling_with_nomic(hybrid_markdown_chunks):
    print("Embedding Docling Markdown with Nomic:")

    try:
        # ✅ Proper Nomic initialization (FIXED)
        embeddings = HuggingFaceEmbeddings(
            model_name="nomic-ai/nomic-embed-text-v1",
            model_kwargs={
                "device": "cpu",
                "trust_remote_code": True
            },
            encode_kwargs={
                "normalize_embeddings": True
            }
        )

        persist_dir = "./chroma_docling_nomic"

        print(f"Embedding {len(hybrid_markdown_chunks)} chunks:")

        # Generate IDs
        ids = [str(uuid.uuid4()) for _ in hybrid_markdown_chunks]

        # ✅ Create Chroma DB
        vector_db = Chroma.from_documents(
            documents=hybrid_markdown_chunks,
            embedding=embeddings,
            ids=ids,
            persist_directory=persist_dir,
            collection_name="docling_nomic_collection"
        )

        print(f"Successfully embedded and saved Chroma database to {persist_dir}")
        return vector_db

    except Exception as e:
        print(f"Error during Nomic embedding: {e}")
        print("⚠️ Switching to fallback embedding model (BGE-M3)...")

        # ✅ FALLBACK (ALWAYS WORKS)
        embeddings = HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cpu"},
            encode_kwargs={"normalize_embeddings": True}
        )

        vector_db = Chroma.from_documents(
            documents=hybrid_markdown_chunks,
            embedding=embeddings,
            persist_directory="./chroma_fallback",
            collection_name="fallback_collection"
        )

        print("✅ Fallback embedding successful!")
        return vector_db


# ✅ RUN
docling_vector_db = embed_docling_with_nomic(hybrid_chunks_docling)

Embedding Docling Markdown with Nomic:


Embedding 482 chunks:
Successfully embedded and saved Chroma database to ./chroma_docling_nomic


## PHASE 4.1: Retrieval, LLM Integration, Chains and Response generation

### Retrieval

A Retriever is a specific interface (a Python object) whose only job is to take a user's string query (e.g., "What is the refund policy?") and return a list of relevant Document objects.

While a Vector Database stores the data, the Retriever is the tool that fetches it and hands it over to the LLM.

By default, .as_retriever() uses standard similarity search. However, you can change its behavior by passing a search_type parameter. Here are the other powerful methods you can use:

1. Maximal Marginal Relevance (MMR)

Code examole: retriever = vector_db.as_retriever(search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20})

Standard similarity search has a flaw: if the top 4 most similar documents all contain the exact same paragraph (maybe duplicated in your data), the LLM gets redundant information.

MMR solves this by balancing relevance and diversity.

First, it fetches a larger pool of documents (e.g., fetch_k=20).

Then, it selects the single most relevant document.

For the remaining k-1 documents, it actively looks for chunks that are relevant to the query but different from the documents it already selected. This gives the LLM a broader context to write a better answer.

2. Similarity Score Threshold

Code example: retriever = vector_db.as_retriever(search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.8, "k": 4})

Standard search will always return k documents, even if they are terrible matches. If the user asks about "SpaceX" but your database is full of "Cooking Recipes," standard search will just return the 4 recipes that are "least different" from SpaceX.

Using a threshold tells the retriever: "Only return documents if they have a similarity score of 80% (0.8) or higher. If you can't find 4 good ones, just return whatever passes the threshold, even if it's only 1 or 2."


#### There are other useful parameters you can pass into the search_kwargs dictionary:

- filter: This allows you to filter by document metadata before the vector search happens.

Example: search_kwargs={"k": 4, "filter": {"year": 2026, "department": "engineering"}}. This is incredibly efficient because it narrows down the search space immediately.

- lambda_mult: (Used only with MMR). This is a number between 0 and 1 that controls the diversity penalty.

0.5 is the default balanced setting.

Closer to 0 forces maximum diversity (very different documents).

Closer to 1 forces maximum relevance (basically regular similarity search).



#### We will use Ollama models for the LLM Integration part.

In [38]:
%pip install langchain-ollama

In [39]:
# Ollama setup on colab

# 1. Install zstd and Ollama
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama with a command that works better for Colab backgrounding
import subprocess
import os
import time
import requests

# This starts the server and pipes output away so it doesn't block the cell
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    preexec_fn=os.setsid # This keeps it running in a separate session
)

# 3. Wait and verify connection
print("Waiting for Ollama to wake up...")
for i in range(10):
    try:
        # Check if the server is actually listening
        res = requests.get("http://localhost:11434")
        if res.status_code == 200:
            print("SUCCESS: Ollama is reachable!")
            break
    except:
        time.sleep(2)
        print(f"Retrying connection... ({i+1}/10)")

# 4. Pull the model
!ollama pull <MODEL_NAME>

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,442 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,934 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu 

In [41]:
hybrid_vector_db = advanced_faiss_indexing_with_huggingface(hybrid_chunks)

Loading embedding model...
Extracting texts and generating raw embeddings for 482 chunks...
Embedding dimension: 1024
Building Flat Index...
Building IVF Index...
Building HNSW Index...
Building PQ Index...

Retrieval Benchmarks:
Flat Index (L2)  Latency: 0.2766 ms | Retrieved Indices: [98 99 63]
IVF Index        Latency: 0.3791 ms | Retrieved Indices: [ 98  99 198]
HNSW Index       Latency: 0.1884 ms | Retrieved Indices: [98 99 63]
PQ Index         Latency: 0.4644 ms | Retrieved Indices: [ 98  99 284]

Wrapping HNSW index into LangChain FAISS VectorStore:
VectorStore saved locally as 'faiss_index'


In [42]:
faiss_retriever = hybrid_vector_db.as_retriever(search_kwargs={"k": 3})

In [ ]:
faiss_retriever = <loader>_vector_db.as_retriever(search_kwargs={"k": 3})

SyntaxError: invalid syntax (2934185723.py, line 1)

### Chains

If LLMs are the engine, Chains are the transmission, steering wheel, and wheels that actually make the car drive.

An LLM by itself just spits out text. To build a useful application (like your RAG system), you need to combine the LLM with other components in a specific sequence. A Chain is exactly what it sounds like: a linked sequence of operations.

The most fundamental Chain in LangChain consists of three links:

- Prompt Template: Takes raw user input and formats it into a structured prompt with instructions.

- LLM: Processes that structured prompt and generates a response.

- Output Parser: Takes the raw text from the LLM and converts it into a usable format (like a Python dictionary, a JSON object, or a list).

#### Choose an appropriate model for this task from https://ollama.com/library and make sure its with base_url ="http://127.0.0.1:11434" as an argument.


In [43]:
!pip install langchain-ollama

In [44]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def build_and_run_rag_chain(vector_db, user_question):

    # ✅ Retriever setup
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    # ✅ Initialize Ollama LLM
    llm = ChatOllama(
        model="llama3",   # make sure you ran: ollama pull llama3
        temperature=0.2
    )

    # ✅ Strong anti-hallucination system prompt
    system_prompt = """
    You are a helpful AI assistant answering questions about a movie screenplay.

    Use ONLY the provided context to answer the question.

    Rules:
    - Do NOT make up information.
    - If the answer is not in the context, say: "I could not find the answer in the provided context."
    - Be clear and concise.

    Context:
    {context}
    """

    # Prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}")
    ])

    # Format retrieved documents
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # ✅ RAG pipeline
    rag_chain = (
        {
            "context": retriever | format_docs,
            "input": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    # Run query
    response = rag_chain.invoke(user_question)

    print("Response:\n", response)

    return response

### TRY OUT ON MORE QUESTIONS PROVIDED IN THE MANUAL.

In [45]:
print(type(hybrid_vector_db))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [46]:
%who

ChatOllama	 ChatPromptTemplate	 Chroma	 DoclingLoader	 Document	 DocumentConverter	 FAISS	 HuggingFaceEmbeddings	 InMemoryDocstore	 
MarkdownHeaderTextSplitter	 PDF_PATH	 PdfFormatOption	 PdfPipelineOptions	 PyMuPDFLoader	 PyPDFLoader	 RecursiveCharacterTextSplitter	 RunnablePassthrough	 StrOutputParser	 
advanced_faiss_indexing_with_huggingface	 build_and_run_rag_chain	 chunk_fixed	 chunk_markdown_hybrid	 chunk_recursive	 custom_converter	 docling_docs	 docling_vector_db	 embed_docling_with_nomic	 
embeddings	 extract_with_docling	 faiss	 faiss_retriever	 fixed_chunks	 hybrid_chunks	 hybrid_chunks_docling	 hybrid_vector_db	 i	 
langchain_extract_pymupdf	 langchain_extract_pypdf	 logging	 np	 os	 pipeline_options	 process	 pymupdf_docs	 pypdf_docs	 
recursive_chunks	 requests	 res	 subprocess	 time	 uuid	 


In [47]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [48]:
!ollama pull llama3

In [49]:
question = "What is the cause of Uncle Vernon's irritation?"

answer = build_and_run_rag_chain(hybrid_vector_db, question)

Response:
 According to the provided context, the cause of Uncle Vernon's irritation appears to be the state of his house being in ruins. He says "And come back to find the house in ruins?" which suggests that he was not expecting this and is upset about it.


In [50]:
question = "What is the cause of Uncle Vernon's irritation?"
answer = build_and_run_rag_chain(docling_vector_db, question)

Response:
 According to the context, Uncle Vernon's irritation is caused by the fact that there are no letters for him today. He exclaims "No damn letters today! No sir. Not one blasted letter!"


## PHASE 4.2: How to add a memory component? (Chat History)

In [72]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# store sessions
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

def build_and_run_rag_chain_memory(vector_db, user_question, session_id="default"):
    # 1. Initialize Retriever [cite: 5, 28]
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    # 2. Initialize LLM
    llm = ChatOllama(model="llama3.2:1b", temperature=0)

    # 3. Define the System Prompt with Grounding [cite: 5, 28]
    system_prompt = (
        "You are an expert on the Harry Potter screenplay. "
        "Use the retrieved context to answer the question. "
        "If you don't know the answer, say you don't know. "
        "Context: {context}"
    )

    # 4. Create Prompt Template with Chat History placeholder
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])

    # 5. Build the internal RAG Chain
    # We use itemgetter or lambda to map inputs for the chain
    rag_chain = (
        {"context": retriever, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    # 6. Wrap chain with memory
    # We specify "history" as the variable name used in the prompt's MessagesPlaceholder
    chain_with_memory = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    # 7. Invoke the chain with the session configuration
    response = chain_with_memory.invoke(
        {"input": user_question},
        config={"configurable": {"session_id": session_id}}
    )

    print("Response:\n", response)

    return response

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


# store sessions
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


def build_and_run_rag_chain_memory(vector_db, user_question, session_id="default"):

    """
    TODO: Take from previous function.
    """
    retriever = None

    llm = None

    system_prompt = None

    prompt = None


    rag_chain = None

    # Wrap chain with memory
    """
    TODO: Wrap the rag_chain with RunnableWithMessageHistory, passing in the get_session_history function to manage chat history for different sessions.
    """
    chain_with_memory = None

    response = chain_with_memory.invoke(
        {"input": user_question},
        config={"configurable": {"session_id": session_id}}
    )

    print("Response:\n", response)

    return response

In [74]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from operator import itemgetter

# store sessions
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

def build_and_run_rag_chain_memory(vector_db, user_question, session_id="default"):
    # 1. Initialize Retriever
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    # 2. Initialize LLM
    llm = ChatOllama(model="llama3.2:1b", temperature=0)

    # 3. Define the System Prompt with Grounding
    # This matches the objective of applying strict system grounding prompts [cite: 143]
    system_prompt = (
        "You are an expert on the Harry Potter screenplay. "
        "Use the retrieved context to answer the question. "
        "If you don't know the answer, say you don't know. "
        "Context: {context}"
    )

    # 4. Create Prompt Template with Chat History placeholder
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])

    # 5. Build the internal RAG Chain using LCEL
    # The error is fixed here by ensuring the context is a joined string of page_contents
    rag_chain = (
        {
            "context": itemgetter("input") | retriever | (lambda docs: "\n\n".join(doc.page_content for doc in docs)),
            "input": itemgetter("input"),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    # 6. Wrap chain with memory
    # Using RunnableWithMessageHistory as per the manual's requirement
    chain_with_memory = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    # 7. Invoke the chain
    response = chain_with_memory.invoke(
        {"input": user_question},
        config={"configurable": {"session_id": session_id}}
    )

    print("Response:\n", response)
    return response

### TRY OUT ON MORE QUESTIONS PROVIDED IN THE MANUAL.

In [77]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# store sessions
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


def build_and_run_rag_chain_memory(vector_db, user_question, session_id="default"):
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})

    llm = ChatOllama(model="llama3", temperature=0)

    system_prompt = """You are an expert assistant for the Harry Potter and the Sorcerer's Stone screenplay.
Answer ONLY based on the retrieved context below. If the answer is not in the context, say 'I cannot find that information in the screenplay.'

Context:
{context}"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])

    def get_context(input_dict):
        return retriever.invoke(input_dict["input"])

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | prompt
        | llm
        | StrOutputParser()
    )

    # Wrap chain with memory
    chain_with_memory = RunnableWithMessageHistory(
        rag_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )

    response = chain_with_memory.invoke(
        {"input": user_question},
        config={"configurable": {"session_id": session_id}}
    )

    print("Response:\n", response)
    return response

In [78]:
build_and_run_rag_chain_memory(docling_vector_db, "Who gives Harry his first birthday cake?", "session1")

Response:
 According to the screenplay, Hagrid gives Harry his first birthday cake, which is a chocolate cake with "Happee Birthdae, Harry" scrawled in green icing.


'According to the screenplay, Hagrid gives Harry his first birthday cake, which is a chocolate cake with "Happee Birthdae, Harry" scrawled in green icing.'

In [79]:
build_and_run_rag_chain_memory(docling_vector_db, "What is written on it?", "session1")

Response:
 I can find that information! According to the context, the message written on the cake is: "Happee Birthdae, Harry".


'I can find that information! According to the context, the message written on the cake is: "Happee Birthdae, Harry".'

## Phase 4.3: Custom Branching Logic & Anti-Hallucination

In [80]:
from langchain_core.runnables import RunnableBranch
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Part 1: Strict System Prompt (Anti-Hallucination)
qa_system_prompt = """You are a strict continuity assistant for the Harry Potter screenplay.
You must answer the user's questions based ONLY on the provided screenplay context below.
If the context does not contain the answer, you must explicitly state: "This information is not present in the screenplay."
Do NOT hallucinate details from the books or movies that are not in the text.

Context:
{context}
"""

# TODO 1: QA ChatPromptTemplate
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# TODO 2: General conversation ChatPromptTemplate
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant. Respond warmly to casual greetings and small talk."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# TODO 3: Routing function
def route_query(info):
    user_input = info.get("input", "").strip().lower()
    greetings = ["hi", "hello", "hey", "howdy", "greetings"]
    word_count = len(user_input.split())
    if any(user_input.startswith(g) for g in greetings) and word_count < 6:
        return "chat"
    return "qa"

faiss_retriever = docling_vector_db.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

llm = ChatOllama(model="llama3", temperature=0)

# TODO 4: RAG QA Chain
qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(faiss_retriever.invoke(x["input"]))
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# TODO 5: Simple Chat Chain
chat_chain = (
    chat_prompt
    | llm
    | StrOutputParser()
)

# TODO 6: RunnableBranch
branch = RunnableBranch(
    (lambda x: route_query(x) == "chat", chat_chain),
    qa_chain  # default fallback
)

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# TODO 7: Wrap with memory
branching_rag_chain = RunnableWithMessageHistory(
    branch,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("--- Testing Phase 4.3: Branching, Grounding & Memory ---")
session_id = "hogwarts_branching_01"

def ask_branching_pipeline(question):
    print(f"\nUser: {question}")
    # TODO 8: Invoke the final chain
    response = branching_rag_chain.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"AI: {response}")

--- Testing Phase 4.3: Branching, Grounding & Memory ---


In [81]:
# Test 1: Chitchat (Bypasses Vector DB, saves compute)
ask_branching_pipeline("Hello! How are you?")

# Test 2: RAG Factual Query
ask_branching_pipeline("Who gives Harry his first birthday cake?")

# Test 3: Anti-Hallucination Test (Should trigger the strict system prompt constraint)
ask_branching_pipeline("Summarize the scene where Harry fights Darth Vader.")


User: Hello! How are you?
AI: Hi there! I'm doing great, thanks for asking! It's always lovely to start the day with a chat. How about you? How's your morning going so far?

User: Who gives Harry his first birthday cake?
AI: According to the screenplay context, Hagrid gives Harry his first birthday cake, which is a chocolate cake with "Happee Birthdae, Harry" scrawled in green icing.

User: Summarize the scene where Harry fights Darth Vader.
AI: This information is not present in the screenplay. There is no scene where Harry fights Darth Vader in the provided context.


## Phase 5.1 Hybrid Search

Hybrid search (H) can be seen as the combination of searches with keyword representation (K) or vector embedding representation (V). It is simply expressed in the following expression.

H = (1-α) K + αV
Where α is a parameter chosen by the user, its value is between 0 and 1. With this parameter, we can give more importance to keyword search or vector search.

Vector search is based on comparing embeddings with measures similar to cosine similarity, which compares the angle between vectors and their relative direction.

Keyword search is based on the use of sparse vectors to search by frequency criteria. This search measures the frequency of words within the document and within the set of documents, the more frequent a word is in a document and the less frequent this word is in the set of documents, the better. Therefore, the retrieved documents contain the searched words and are more unique within the set of documents.



In [82]:
%pip install rank_bm25 sentence-transformers

In [84]:
# Run this to find your chunk variable names
%whos list

Variable                Type    Data/Info
-----------------------------------------
docling_docs            list    n=1
docs                    list    n=4
fixed_chunks            list    n=674
hybrid_chunks           list    n=482
hybrid_chunks_docling   list    n=482
pypdf_docs              list    n=134
recursive_chunks        list    n=432


In [86]:
bm25_retriever = BM25Retriever.from_documents(hybrid_chunks_docling)

In [88]:
from langchain_community.retrievers import BM25Retriever

print("--- Phase 5.1: Initializing Custom Hybrid Search (RRF) ---")

# TODO 1: Initialize the BM25 Keyword Retriever
print("Building BM25 Keyword Retriever...")
bm25_retriever = BM25Retriever.from_documents(hybrid_chunks_docling)
bm25_retriever.k = 5

faiss_retriever = docling_vector_db.as_retriever(search_kwargs={"k": 5})

# Custom Hybrid Retrieval Function (Reciprocal Rank Fusion)
def custom_hybrid_retrieve(query, top_k=5, bm25_weight=0.3, faiss_weight=0.7):
    """
    Executes a hybrid search using both BM25 and FAISS, then fuses the results
    using the Reciprocal Rank Fusion (RRF) algorithm.
    """

    # TODO 2: Fetch documents from both retrievers
    bm25_docs = bm25_retriever.invoke(query)
    faiss_docs = faiss_retriever.invoke(query)

    # Dictionaries to keep track of scores and document objects
    doc_scores = {}
    doc_map = {}

    # Standard RRF constant
    c = 60

    # TODO 3: Score BM25 documents based on their rank
    for rank, doc in enumerate(bm25_docs):
        key = doc.page_content
        rrf_score = bm25_weight * (1 / (rank + c))
        doc_scores[key] = doc_scores.get(key, 0) + rrf_score
        doc_map[key] = doc

    # TODO 4: Score FAISS documents based on their rank
    for rank, doc in enumerate(faiss_docs):
        key = doc.page_content
        rrf_score = faiss_weight * (1 / (rank + c))
        doc_scores[key] = doc_scores.get(key, 0) + rrf_score
        doc_map[key] = doc

    # TODO 5: Sort by highest fused score
    sorted_keys = sorted(doc_scores.keys(), key=lambda k: doc_scores[k], reverse=True)

    # Return Top K document objects
    return [doc_map[k] for k in sorted_keys[:top_k]]

print("Custom Hybrid Retriever successfully created")

# --- Quick test ---
test_query = "Oculus Reparo"
print(f"\nTesting hybrid retrieval for: '{test_query}'")
hybrid_results = custom_hybrid_retrieve(test_query)
for i, doc in enumerate(hybrid_results):
    print(f"\n[Result {i+1}] Score key: {list(doc_scores.items())[i] if 'doc_scores' in dir() else ''}")
    print(doc.page_content[:300])

--- Phase 5.1: Initializing Custom Hybrid Search (RRF) ---
Building BM25 Keyword Retriever...
Custom Hybrid Retriever successfully created

Testing hybrid retrieval for: 'Oculus Reparo'

[Result 1] Score key: 
## HERMIONE  (CONT'D)

Goodness. You're Harry Potter, aren't· you? I know  all about you, of course. I was  doing a little recreational reading and you're in Modern  Magical History, The Rise and  Fall of the Dark  Arts and Great

. Wizarding Events of the 20th Century.

Arn I'?

HARRY

## HERMIONE


[Result 2] Score key: 
·  This one's lost its head.

[Result 3] Score key: 
But  that's Duo.ley's old uniform. It'll fit me  like bits of old elephant skin.

[Result 4] Score key: 
of Uncle Vernon's thick black shoes pa~ing back and forth.

[Result 5] Score key: 
We could phone  Yvonne.


In [89]:
test_query = "What exact words does the Sorting Hat say before placing Harry?"
print(f"\nTesting Query: '{test_query}'")
hybrid_docs = custom_hybrid_retrieve(test_query)

for i, doc in enumerate(hybrid_docs[:2]):
    print(f"Result {i+1}:\n{doc.page_content[:150]}...\n")


Testing Query: 'What exact words does the Sorting Hat say before placing Harry?'
Result 1:
## SORTING HAT  ·

Oh,  you  may  not think I'm pretty But don't judge on what  you see I'll eat myself if you can  fina A  smarter hat than  me. Ther...

Result 2:
## SORTING  HAT

Hmmrn. Difficult. Very  Difficult. Plenty of courage, I see. Not  a  bad mind  either. There's talent, oh  yes, and  a  thirst to pro...



## PHASE 5.2: Re-Ranking

Cross encoder models take pairs of (query, document) and output a relevance score, which is perfect for our reranking step. By using a cross-encoder, we allow the model to deeply understand the interaction between the query and each candidate document, leading to more accurate relevance scoring compared to bi-encoder models that encode query and documents separately.

In [90]:
from sentence_transformers import CrossEncoder
import numpy as np

print("--- Phase 5.2: Initializing Custom Cross-Encoder Re-Ranker ---")

# 1. Initialize Cross-Encoder Model
print("Loading BAAI/bge-reranker-base...")
cross_encoder_model = CrossEncoder('BAAI/bge-reranker-base')

# 2. Reranking Function
def custom_reranked_retrieve(query, initial_k=10, final_k=3):
    # Step A: Get initial candidates from hybrid retriever
    candidates = custom_hybrid_retrieve(query, top_k=initial_k)

    if not candidates:
        return []

    # Step B: Prepare query-document pairs for cross-encoder
    pairs = [[query, doc.page_content] for doc in candidates]

    # Step C: Predict relevance scores
    scores = cross_encoder_model.predict(pairs)

    # Step D: Sort by highest score descending
    sorted_indices = np.argsort(scores)[::-1]

    # Step E: Return top final_k documents
    reranked_docs = [candidates[i] for i in sorted_indices[:final_k]]

    return reranked_docs

print("Custom Re-Ranker initialized!")

# Quick test
test_query = "What spell does Hermione use on the troll?"
print(f"\nTesting re-ranker for: '{test_query}'")
reranked_results = custom_reranked_retrieve(test_query)
for i, doc in enumerate(reranked_results):
    print(f"\n[Re-ranked Result {i+1}]")
    print(doc.page_content[:300])

--- Phase 5.2: Initializing Custom Cross-Encoder Re-Ranker ---
Loading BAAI/bge-reranker-base...
Custom Re-Ranker initialized!

Testing re-ranker for: 'What spell does Hermione use on the troll?'

[Re-ranked Result 1]
-

-

.

103.

CUT TO:

104

~

---

*

*

*

.

...

--

--.

.:·---=·.

()

u

## HERMIONE' S  POV  ·

···  as the troll advances directly toward her.

She  dashes into a stall, bolts the door~.  Trembling, she peers upward ···  watching as ···  the troll's face appears over the top, looking in.

[Re-ranked Result 2]
·..  THUNK!  It drops smack  on  the troll's head. Wobbling, the troll releases its grip on  Harry's leg and ·.·

... drops him  hard to the floor·. Harry peers up. The  troll wobbles one las.t time and starts to fall ···  directly on  top of

Harry. Quickly, Harry rolls away.· ..

... just before t

[Re-ranked Result 3]
Miss Granger!

## HERMIONE

I  went looking for the troll. I've read ·about them  and  thought I  .could handle it. But I  was  wrong. If 

In [91]:
print(f"\nRe-Ranking Top Results for '{test_query}'...")
reranked_docs = custom_reranked_retrieve(test_query)

for i, doc in enumerate(reranked_docs):
    print(f"\n--- Reranked Result {i+1} ---")
    print(f"{doc.page_content[:250]}...")


Re-Ranking Top Results for 'What spell does Hermione use on the troll?'...

--- Reranked Result 1 ---
-

-

.

103.

CUT TO:

104

~

---

*

*

*

.

...

--

--.

.:·---=·.

()

u

## HERMIONE' S  POV  ·

···  as the troll advances directly toward her.

She  dashes into a stall, bolts the door~.  Trembling, she peers upward ···  watching as ···  th...

--- Reranked Result 2 ---
·..  THUNK!  It drops smack  on  the troll's head. Wobbling, the troll releases its grip on  Harry's leg and ·.·

... drops him  hard to the floor·. Harry peers up. The  troll wobbles one las.t time and starts to fall ···  directly on  top of

Harry....

--- Reranked Result 3 ---
Miss Granger!

## HERMIONE

I  went looking for the troll. I've read ·about them  and  thought I  .could handle it. But I  was  wrong. If Harry and  Ron  hadn't come  along  ... I'd be dead.

Ron  drops his wand, stunned by  Hermione's lie.

## PROFE...


## PHASE 5.3. Basic Query Expansion

In [92]:
llm = ChatOllama(model="llama3", temperature=0, base_url="http://127.0.0.1:11434")

In [93]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("--- Phase 5.3: Initializing Query Expansion ---")

# 1. Query Expansion Prompt
expansion_prompt = PromptTemplate.from_template(
    """You are an expert on the Harry Potter universe and screenplay terminology.
Your task is to expand a short, vague user query into a richer search query.
Add relevant character names, magical spell names, location names, and thematic synonyms
that would help retrieve the most relevant screenplay passages.
Output ONLY the expanded query as a single sentence. No explanations, no preamble.

Original Query: {question}
Expanded Query:"""
)

# 2. Build the Expansion Chain
query_expansion_chain = expansion_prompt | llm | StrOutputParser()

# Testing the query expansion
lazy_query = "Harry is sad"
expanded = query_expansion_chain.invoke({"question": lazy_query})
print(f"Original Query: {lazy_query}")
print(f"Expanded Query: {expanded}")

# 3. Integrate into the Retrieval Function
def retrieve_with_expansion(query):
    print("\n[System] Expanding query...")
    expanded_query = query_expansion_chain.invoke({"question": query})
    print(f"[System] Searching database for: {expanded_query}")
    docs = custom_reranked_retrieve(expanded_query)
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Final Advanced QA Chain
advanced_qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: retrieve_with_expansion(x["input"])
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

print("\n--- Testing the fully integrated Advanced RAG Pipeline ---")

final_response = advanced_qa_chain.invoke({
    "input": "What is the cause of Uncle Vernon's irritation?",
    "history": []
})

print(f"\nFinal AI Response:\n{final_response}")

--- Phase 5.3: Initializing Query Expansion ---
Original Query: Harry is sad
Expanded Query: Harry Potter's emotional state is characterized by feelings of sorrow or melancholy, similar to those experienced after the death of Albus Dumbledore at Hogwarts School of Witchcraft and Wizardry, possibly involving the use of a Patronus Charm or a conversation with Ron Weasley or Hermione Granger.

--- Testing the fully integrated Advanced RAG Pipeline ---

[System] Expanding query...
[System] Searching database for: What are the motivations behind Uncle Vernon's frustration when Harry Potter, Hermione Granger, and Ron Weasley arrive at his home, specifically in scenes where he uses spells like "Impedimenta" or "Reducto", possibly triggered by the presence of Muggles, Hogwarts' influence, or the trio's unconventional behavior?

Final AI Response:
According to the screenplay context, Uncle Vernon's irritation is caused by the "BOOM!" sounds and the door shuddering. It seems that something has h

# CHOOSE ANY 2 IN PHASE 6

## PHASE 6.1 HyDE (Hypothetical Document Embeddings)

HyDE uses a Language Learning Model, to create a theoretical document when responding to a query, as opposed to using the query and its computed vector to directly seek in the vector database.

It goes a step further by using an unsupervised encoder learned through contrastive methods. This encoder changes the theoretical document into an embedding vector to locate similar documents in a vector database.

Rather than seeking embedding similarity for questions or queries, it focuses on answer-to-answer embedding similarity.


In [94]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("--- Phase 6.1: HyDE (Hypothetical Document Embeddings) ---")

# TODO 1: HyDE PromptTemplate
hyde_prompt = PromptTemplate.from_template(
    """You are a screenplay writer for Harry Potter and the Sorcerer's Stone.
Write a short, fictional screenplay scene (3-5 sentences) that directly answers the question below.
Write ONLY the scene text. Do NOT repeat the question or add any explanation.

Question: {question}
Hypothetical Scene:"""
)

# TODO 2: HyDE Generator Chain
hyde_generator = hyde_prompt | llm | StrOutputParser()

# 3. Custom HyDE Retrieval Function
def retrieve_hyde(query):
    print("\n[HyDE] Generating hypothetical scene...")

    # TODO 3: Generate the fake scene
    fake_scene = hyde_generator.invoke({"question": query})
    print(f"[HyDE] Fake Scene Snippet: {fake_scene[:150]}...")

    print("[HyDE] Searching Vector DB using the fake scene...")

    # TODO 4: Search Vector DB using the fake scene
    docs = faiss_retriever.invoke(fake_scene)

    # TODO 5: Format and return
    return "\n\n".join(doc.page_content for doc in docs)

# TODO 6: Final HyDE QA Chain
hyde_qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: retrieve_hyde(x["input"])
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Test HyDE
print("\n--- Testing HyDE Pipeline ---")
hyde_response = hyde_qa_chain.invoke({
    "input": "How does Harry feel when he first sees Hogwarts?",
    "history": []
})
print(f"\nHyDE Response:\n{hyde_response}")

--- Phase 6.1: HyDE (Hypothetical Document Embeddings) ---

--- Testing HyDE Pipeline ---

[HyDE] Generating hypothetical scene...
[HyDE] Fake Scene Snippet: INT. HOGWARTS EXPRESS - DAY

The train rumbles to a stop, and Harry's eyes widen as he peers out the window. The castle looms before him, its turrets ...
[HyDE] Searching Vector DB using the fake scene...

HyDE Response:
According to the screenplay, when Harry peers out of his window, his face is calm. Peaceful.


In [95]:
# Test HyDE
print("\nTesting HyDE Pipeline:")
hyde_response = hyde_qa_chain.invoke({
    "input": "How does Harry feel when his wand chooses him at Ollivanders?",
    "history": []
})
print(f"\nHyDE Final Answer:\n{hyde_response}")


Testing HyDE Pipeline:

[HyDE] Generating hypothetical scene...
[HyDE] Fake Scene Snippet: INT. OLLIVANDER'S WIZARDING WOODS - DAY

Harry's eyes widened as the wand began to glow, hovering inches from his hand. He felt an electric thrill run...
[HyDE] Searching Vector DB using the fake scene...

HyDE Final Answer:
According to the screenplay context provided, it is actually Barry who extends his arm and his hand trembles as the shop's tiny bell rings. The pages of a book flutter on the counter, and Barry's hair feathers off his forehead, showing his scar. Astounded, Barry smiles.

There is no mention of Harry's feelings or emotions during this scene at Ollivander's.


## PHASE 6.2: RAG-Fusion

RAG-Fusion combines RAG and reciprocal rank fusion (RRF) by generating multiple queries, reranking them with reciprocal scores and fusing the documents and scores.

Through manually evaluating answers on accuracy, relevance, and comprehensiveness, it was found that RAG-Fusion was able to provide accurate and comprehensive answers due to the generated queries contextualizing the original query from various perspectives.

In [96]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("--- Phase 6.2: RAG-Fusion ---")

# TODO 1: Multi-Query Prompt
multi_query_prompt = PromptTemplate.from_template(
    """You are an AI assistant helping search a Harry Potter screenplay database.
Generate exactly 4 different search query variations for the question below.
Each variation should approach the question from a different angle using different keywords.
Output ONLY the 4 queries, one per line, with no numbering, bullets, or extra text.

Question: {question}
Query Variations:"""
)

# TODO 2: Query Generator Chain
query_generator = multi_query_prompt | llm | StrOutputParser()

def rag_fusion_retrieve(query, top_k=3):
    print(f"\n[RAG-Fusion] Generating query variations for: '{query}'")

    # TODO 3: Generate query variations
    raw_output = query_generator.invoke({"question": query})
    queries = [q.strip() for q in raw_output.split('\n') if q.strip()]
    queries.append(query)  # include original query
    print(f"[RAG-Fusion] Generated queries: {queries}")

    print(f"[RAG-Fusion] Executing {len(queries)} parallel searches...")

    doc_scores = {}
    doc_map = {}
    c = 60

    # TODO 4: Parallel searches with RRF scoring
    for q in queries:
        results = faiss_retriever.invoke(q)
        for rank, doc in enumerate(results):
            key = doc.page_content
            rrf_score = 1 / (rank + c)
            doc_scores[key] = doc_scores.get(key, 0) + rrf_score
            doc_map[key] = doc

    # TODO 5: Sort and select Top K
    sorted_keys = sorted(doc_scores.keys(), key=lambda k: doc_scores[k], reverse=True)
    fused_docs = [doc_map[k] for k in sorted_keys[:top_k]]

    return "\n\n".join(doc.page_content for doc in fused_docs)

# Build RAG-Fusion QA Chain
rag_fusion_chain = (
    RunnablePassthrough.assign(
        context=lambda x: rag_fusion_retrieve(x["input"])
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Test RAG-Fusion
print("\n--- Testing RAG-Fusion Pipeline ---")
fusion_response = rag_fusion_chain.invoke({
    "input": "How does the screenplay build suspense around the Sorcerer's Stone?",
    "history": []
})
print(f"\nRAG-Fusion Response:\n{fusion_response}")

--- Phase 6.2: RAG-Fusion ---

--- Testing RAG-Fusion Pipeline ---

[RAG-Fusion] Generating query variations for: 'How does the screenplay build suspense around the Sorcerer's Stone?'
[RAG-Fusion] Generated queries: ["How do plot twists and red herrings contribute to building suspense in the Sorcerer's Stone screenplay?", "What techniques are used to create tension and anticipation throughout the story of the Sorcerer's Stone screenplay?", "Can you identify scenes or moments where the screenplay uses foreshadowing, misdirection, or cliffhangers to build suspense around the Sorcerer's Stone?", "How does the screenplay use character motivations, relationships, and backstories to create suspense and uncertainty surrounding the Sorcerer's Stone?", "How does the screenplay build suspense around the Sorcerer's Stone?"]
[RAG-Fusion] Executing 5 parallel searches...

RAG-Fusion Response:
The screenplay builds suspense around the Sorcerer's Stone by gradually revealing information and creating 

In [98]:
# TODO 6: Build RAG-Fusion QA Chain
fusion_qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: rag_fusion_retrieve(x["input"])
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Test RAG-Fusion
print("\nTesting RAG-Fusion Pipeline:")
fusion_response = fusion_qa_chain.invoke({
    "input": "What gift does Harry receive for Christmas?",
    "history": []
})

print(f"\nRAG-Fusion Final Answer:\n{fusion_response}")


Testing RAG-Fusion Pipeline:

[RAG-Fusion] Generating query variations for: 'What gift does Harry receive for Christmas?'
[RAG-Fusion] Generated queries: ['Harry Potter gifts received on Christmas', 'Christmas presents given to Harry Potter character', 'What is the present Harry receives at Christmas time', 'Gifts bestowed upon Harry Potter during holiday season', 'What gift does Harry receive for Christmas?']
[RAG-Fusion] Executing 5 parallel searches...

RAG-Fusion Final Answer:
According to the screenplay, Harry receives several gifts:

1. A package from Hagrid containing a wooden flute.
2. A Weasley sweater from his mother (Mrs. Potter).
3. A mysterious, shiny, and translucent object that turns out to be some kind of cloak.

Additionally, he also receives a long, thin package delivered by Hedwig the owl, which contains a sleek mahogany broomstick (specifically, a Nimbus 2000).


## PHASE 6.3: CRAG (Corrective RAG)

A lightweight retrieval evaluator is introduced to assess the overall quality of retrieved documents for a given query.

This evaluator is a crucial component in Retrieval-Augmented Generation (RAG), contributing to informative generation by reviewing and evaluating the relevance and reliability of retrieved documents.


In [ ]:
print("--- Phase 6.3: CRAG (Corrective RAG) ---")

# TODO 1: Create the Auditor Prompt
# It must look at the {context} and the {question} and answer ONLY 'yes' or 'no'.
grader_prompt = None

# TODO 2: Build the Grader Chain
grader_chain = None

# The CRAG Routing Logic
def crag_retrieve_and_grade(query):
    """
    Retrieves documents, asks the LLM to grade their relevance, and decides
    whether to pass the context forward or trigger a fallback.
    """
    print("\n[CRAG] Retrieving documents...")

    # TODO 3: Retrieve and format documents
    context_str = ""

    print("[CRAG] Grading relevance...")

    # TODO 4: Invoke the grader
    grade = ""

    # TODO 5: Route based on the grade
    if "yes" in grade:
        pass # Replace with logic
    else:
        pass # Replace with logic


In [ ]:
# 3. Build CRAG QA Chain
def crag_generate(inputs):
    """
    The final generation step that acts differently depending on the Grader's output.
    """
    context = inputs["context"]

    # TODO 6: Handle the Fallback
    # Hint: If the context is exactly "FALLBACK_TRIGGERED", return a string explaining that the database doesn't have the info and a Web Search would be triggered here.
    if context == "FALLBACK_TRIGGERED":
        return ""

    # TODO 7: Run Standard Generation
    generation_chain = None
    return ""

crag_qa_chain = (
    RunnablePassthrough.assign(context=lambda x: crag_retrieve_and_grade(x["input"]))
    | crag_generate
)

In [ ]:
print("\nTesting CRAG 1:")
print(crag_qa_chain.invoke({"input": "What spell does Hermione use to fix Harry's glasses?"}))

print("\nTesting CRAG 2:")
print(crag_qa_chain.invoke({"input": "When does Harry meet Sirius Black?"}))

## Phase 7 - RAG Evaluation Metrics & LLM-as-a-Judge

## Phase 7.1: Dynamic IR Metrics

In traditional Information Retrieval (IR), we use a hardcoded dataset of ground-truth answers to evaluate our search engine. In modern Advanced RAG, we use an **LLM-as-a-Judge** to dynamically read the retrieved chunks and score them as `1` (Relevant) or `0` (Irrelevant) on the fly.

Once we have our dynamic `[1, 0, 1]` scores, we will calculate four foundational IR metrics:


### 1. Precision@K
Measures the "noise" in our retrieval. Out of the $K$ documents we retrieved, what percentage were actually relevant?
* **Formula:** $Precision@K = \frac{\text{Number of Relevant Documents in Top K}}{K}$

### 2. Recall@K (Binary Hit Rate)
True recall requires knowing *every* relevant document in the entire database, which is impossible in dynamic RAG. Instead, we use **Hit Rate** (Binary Recall): Did our retriever find *at least one* relevant document to answer the user's question?
* **Formula:** $Recall@K = 1 \text{ if sum(relevance\_scores)} > 0 \text{ else } 0$

### 3. Mean Reciprocal Rank (MRR)
MRR evaluates the *position* of the first relevant document. Users rarely scroll past the first few results. MRR heavily penalizes systems if the first relevant answer is buried at the bottom.
* **Formula:** $MRR = \frac{1}{\text{Rank of the FIRST relevant document}}$ *(If no relevant docs, MRR = 0)*

### 4. Normalized Discounted Cumulative Gain (NDCG@K)
NDCG is the industry standard for search engines. It accounts for multiple relevant documents and applies a logarithmic penalty for being further down w.r.t to a relevant document which appears in the ranking.
* **Formula:** $NDCG@K = \frac{DCG}{IDCG}$
  * $DCG = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}$ *(where $rel_i$ is the relevance score at rank $i$)*
  * $IDCG = \text{The absolute best possible DCG if all relevant docs were sorted at the very top.}$

In [99]:
import math
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("--- Phase 7.1: Dynamic IR Metrics (LLM-as-a-Judge) ---")

test_queries = [
    "What were the different names Voldemort was called by?",
    "How do Harry, Ron, and Hermione become friends?",
    "What does the centaur warn Harry about in the forest?"
]

# 1. Setup the LLM Judge for Relevance Grading
# TODO: Create a PromptTemplate that asks the LLM to act as a strict relevance grader. It must take {context} and {question} as inputs, and output ONLY '1' (yes) or '0' (no).
relevance_prompt = None

# TODO: Create the grading chain by piping the prompt to your LLM and a StrOutputParser()
relevance_judge = None

k = 3
metrics = {"Precision@K": [], "Recall@K": [], "MRR": [], "NDCG@K": []}

for query in test_queries:
    print(f"\nEvaluating Query: '{query}'")

    # 2. Retrieve Documents
    # TODO: Use your custom_hybrid_retrieve function from Phase 5 to fetch the top_k docs for this query.
    retrieved_docs = []

    # 3. Generate Dynamic Relevance Scores
    relevance_scores = []
    for doc in retrieved_docs:
        # TODO: Invoke your relevance_judge chain with the doc's content and the query. Parse the output to append an integer 1 or 0 to relevance_scores.
        pass

    print(f"Dynamic Relevance Scores: {relevance_scores}")

    # TODO: Calculate the 4 IR Metrics based on the formulae defined for the 4 IR metrics

    # 4. Precision@K
    precision = 0.0
    metrics["Precision@K"].append(precision)

    # 5. Recall@K (Hit Rate)
    recall = 0.0
    metrics["Recall@K"].append(recall)

    # 6. Mean Reciprocal Rank (MRR)
    mrr = 0.0
    metrics["MRR"].append(mrr)

    # 7. NDCG@K
    dcg = 0.0
    idcg = 1.0
    ndcg = 0.0
    metrics["NDCG@K"].append(ndcg)

--- Phase 7.1: Dynamic IR Metrics (LLM-as-a-Judge) ---

Evaluating Query: 'What were the different names Voldemort was called by?'
Dynamic Relevance Scores: []

Evaluating Query: 'How do Harry, Ron, and Hermione become friends?'
Dynamic Relevance Scores: []

Evaluating Query: 'What does the centaur warn Harry about in the forest?'
Dynamic Relevance Scores: []


In [100]:
# Print Final Averages
print(f"\n--- Average Retriever Performance (Top-{k}) ---")
print(f"Precision@{k}: {np.mean(metrics['Precision@K']):.2f}")
print(f"Recall@{k}:    {np.mean(metrics['Recall@K']):.2f} (Hit Rate)")
print(f"MRR:          {np.mean(metrics['MRR']):.2f}")
print(f"NDCG@{k}:       {np.mean(metrics['NDCG@K']):.2f}")


--- Average Retriever Performance (Top-3) ---
Precision@3: 0.00
Recall@3:    0.00 (Hit Rate)
MRR:          0.00
NDCG@3:       0.00


## Phase 7.2: TruLens Evaluation

While IR metrics (e.g. Precision@k, NDCG@k) evaluate the *Retriever*, we also need to evaluate the *Generator* (the LLM). To do this, we use 3 factors to evaluate the Generator: Context Relevance, Groundedness and Output Relevance .

Instead of traditional math formulas, we use an **LLM-as-a-Judge** to score three critical relationships on a scale of 0 to 10, which we then normalize to a $0.0 - 1.0$ scale.

### 1. Context Relevance
Evaluates the relationship between the **User Query** and the **Retrieved Context**. Did the retriever fetch information that actually answers the question, or is it just noise?
* **Formula:** $Context\_Relevance = \frac{\text{LLM\_Score(Query, Context)}}{10}$

### 2. Groundedness/Anti-Hallucination
Evaluates the relationship between the **Retrieved Context** and the **Generated Answer**. Is the LLM's answer strictly supported by the text we provided, or did it hallucinate outside information?
* **Formula:** $Groundedness = \frac{\text{LLM\_Score(Context, Answer)}}{10}$

### 3. Output Relevance
Evaluates the relationship between the **User Query** and the **Generated Answer**. Did the final output actually answer the user's question, or did it go off on a tangent?
* **Formula:** $Answer\_Relevance = \frac{\text{LLM\_Score(Query, Answer)}}{10}$

In this phase, you will write the strict prompts to force the LLM to act as this judge, and use TruLens to automate the grading loop.

In [101]:
%pip install trulens-eval trulens-apps-langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.6/305.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1


In [102]:
import numpy as np
import re
import ast
import time

from trulens_eval import Tru
from trulens.core import Metric
from trulens.apps.custom import instrument

try:
    from trulens.apps.custom import TruApp
except ImportError:
    from trulens.apps.custom import TruCustomApp as TruApp

print("--- Phase 7.2: TruLens Evaluation - LLM-as-a-Judge ---")

# 1. Initialize TruLens
tru = Tru()
tru.reset_database()

# 2. Helper to safely parse scores
def parse_score(llm_output):
    text = getattr(llm_output, 'content', str(llm_output))
    matches = re.findall(r'\d+', text)
    if matches:
        return min(max(float(matches[0]) / 10.0, 0.0), 1.0)
    return 0.0

# 3. Helper to parse TruLens stringified output
def extract_data(response):
    if isinstance(response, dict):
        return response
    if isinstance(response, str):
        try:
            return ast.literal_eval(response)
        except (ValueError, SyntaxError):
            pass
    return {"answer": str(response), "context": "Context not found."}

# 4. Custom Feedback Functions
def custom_context_relevance(query: str, response: str) -> float:
    data = extract_data(response)
    prompt = f"""You are a strict relevance grader for a Harry Potter screenplay RAG system.
Rate how relevant the following Context is to answering the Question on a scale of 0 to 10.
0 = Completely irrelevant. 10 = Perfectly relevant and directly answers the question.
Output ONLY a single integer between 0 and 10. No explanation, no text, just the number.

Question: {query}
Context: {data.get('context', 'No context found.')}
Score:"""
    return parse_score(llm.invoke(prompt))

def custom_groundedness(query: str, response: str) -> float:
    data = extract_data(response)
    prompt = f"""You are a strict groundedness grader for a Harry Potter screenplay RAG system.
Rate how well the Answer is grounded in and supported by the provided Context on a scale of 0 to 10.
0 = The answer is completely fabricated or contradicts the context. 10 = Every claim in the answer is directly supported by the context.
Output ONLY a single integer between 0 and 10. No explanation, no text, just the number.

Context: {data.get('context', 'No context found.')}
Answer: {data.get('answer', 'No answer found.')}
Score:"""
    return parse_score(llm.invoke(prompt))

def custom_answer_relevance(query: str, response: str) -> float:
    data = extract_data(response)
    prompt = f"""You are a strict answer relevance grader for a Harry Potter screenplay RAG system.
Rate how relevant and useful the Answer is in responding to the Question on a scale of 0 to 10.
0 = The answer is completely off-topic. 10 = The answer directly and fully addresses the question.
Output ONLY a single integer between 0 and 10. No explanation, no text, just the number.

Question: {query}
Answer: {data.get('answer', 'No answer found.')}
Score:"""
    return parse_score(llm.invoke(prompt))

# 5. Custom App Wrapper
class HarryPotterRAGApp:
    @instrument
    def __call__(self, query: str) -> dict:
        # Get top 3 docs via hybrid retriever
        docs = custom_hybrid_retrieve(query, top_k=3)

        # Combine page content into single context string
        context_str = "\n\n".join(doc.page_content for doc in docs)

        # Get final answer from advanced_qa_chain
        answer_str = advanced_qa_chain.invoke({
            "input": query,
            "history": []
        })

        return {"answer": answer_str, "context": context_str}

rag_app = HarryPotterRAGApp()

# 6. Metric Objects
f_context_relevance = Metric(
    name="Context Relevance",
    implementation=custom_context_relevance
).on_input_output()

f_groundedness = Metric(
    name="Groundedness",
    implementation=custom_groundedness
).on_input_output()

f_answer_relevance = Metric(
    name="Answer Relevance",
    implementation=custom_answer_relevance
).on_input_output()

# 7. Wrap with TruLens
tru_recorder = TruApp(
    rag_app,
    app_id="Harry_Potter_Advanced_RAG",
    feedbacks=[f_context_relevance, f_groundedness, f_answer_relevance]
)

# 8. Run tests through TruLens recorder
test_queries = [
    "What is the cause of Uncle Vernon's irritation?",
    "How does Harry's life change from beginning to end?",
    "Compare and contrast the way Harry Potter is perceived by his uncle and aunt versus wizards and witches around the world?"
]

print("Executing queries through the pipeline...")
with tru_recorder as recording:
    for query in test_queries:
        print(f"\n[Query]: {query}")
        result = rag_app(query)
        print(f"[Answer]: {result['answer'][:200]}...")

print("\nWaiting for 30 seconds for the LLM Judge to finish grading...")
time.sleep(30)

print("\n--- Final TruLens Evaluation Scores ---")
df = tru.get_leaderboard(app_ids=["Harry_Potter_Advanced_RAG"])

if not df.empty:
    display_df = df[['Context Relevance', 'Groundedness', 'Answer Relevance', 'latency', 'total_cost']]
    display(display_df)
else:
    print("Scores still calculating. Run `display(tru.get_leaderboard())` in a new cell shortly.")

/tmp/ipykernel_1688/386990247.py:6: DeprecationWarning: The `trulens_eval` module is deprecated. See https://www.trulens.org/component_guides/other/trulens_eval_migration/ for instructions on migrating to `trulens.*` modules.
  from trulens_eval import Tru


--- Phase 7.2: TruLens Evaluation - LLM-as-a-Judge ---
🦑 Initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `TruSession` to prevent this.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


✅ experimental Feature.OTEL_TRACING enabled.
🔒 experimental Feature.OTEL_TRACING is enabled and cannot be changed.


Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


Executing queries through the pipeline...

[Query]: What is the cause of Uncle Vernon's irritation?

[System] Expanding query...
[System] Searching database for: What are the underlying reasons behind Uncle Vernon's frustration and annoyance, particularly in relation to Harry Potter's presence at Number Four Privet Drive, influenced by his own insecurities and fear of being embarrassed in front of his neighbors, such as Petunia Dursley and Dudley Dursley?
[Answer]: According to the screenplay, there is no indication of Uncle Vernon being irritated. He appears to be reading the Daily Mail and asking Barry to bring in his coffee, which suggests a normal or calm de...

[Query]: How does Harry's life change from beginning to end?

[System] Expanding query...
[System] Searching database for: How do Harry Potter's experiences with spells like Expecto Patronum and Protego, his relationships with characters like Ron Weasley, Hermione Granger, Ginny Weasley, Luna Lovegood, and Draco Malfoy, and

In [103]:
# Try this after sometime if above cell doesn't show scores due to LLM grading latency
display(tru.get_leaderboard())

,,Answer Relevance,Context Relevance,Groundedness,latency,total_cost
app_name,app_version,,,,,
Harry_Potter_Advanced_RAG,base,0.966667,0.0,0.6,11.956576,0.0
